In [1]:
import pandas as pd
import itertools
import random
import numpy as np
import os
import pandas as pd
import random
from itertools import product
from rdkit import Chem

In [2]:
def is_valid_smiles(smiles):
    if pd.isna(smiles):
        return False

    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol is not None
    except:
        return False

In [3]:
file_path = './raw_data/ChG-Miner_miner-chem-gene.tsv'
pos_df = pd.read_csv(file_path, sep="\t", dtype=str)
pos_df.columns = ["DRUGID", "UNIPROTID"]

pos_df["DRUGID"] = pos_df["DRUGID"].str.strip()
pos_df["UNIPROTID"] = pos_df["UNIPROTID"].str.strip()

pos_df = pos_df.drop_duplicates(subset=['DRUGID', 'UNIPROTID'])
print(f"Total pairs: {len(pos_df)}", pos_df.columns)

drug_df = pd.read_excel("./raw_data/DrugInfo.xlsx", dtype=str)
drug_df = drug_df[['DRUGID', 'DRUGNAME', 'SMILES', 'INCHIKEY']]

prot_df = pd.read_excel("../UniprotKB/uniprotkb_2026_04_01.xlsx")
prot_df = prot_df[['Entry', 'Gene Names', 'Sequence']]
prot_df.columns = ["UNIPROTID", 'GENENAME', "SEQUENCE"]
prot_df["UNIPROTID"] = prot_df["UNIPROTID"].astype("string").str.strip()

pos_pair_df = pd.merge(pos_df, drug_df, on='DRUGID', how='inner')
pos_pair_df = pd.merge(pos_pair_df, prot_df, on='UNIPROTID', how='inner')

print('Validating SMILES...')
pos_pair_df['valid_smiles'] = pos_pair_df['SMILES'].apply(is_valid_smiles)

invalid_df = pos_pair_df[~pos_pair_df['valid_smiles']]
valid_df = pos_pair_df[pos_pair_df['valid_smiles']].copy()

print(f'Valid pairs: {len(valid_df)}')
print(f'Invalid SMILES pairs removed: {len(invalid_df)}')

# ===== 6. 删除辅助列 =====
valid_df = valid_df.drop(columns=['valid_smiles'])

# ===== 7. 保存 =====
save_dir = "./final_data"
os.makedirs(save_dir, exist_ok=True)

valid_df.to_excel(os.path.join(save_dir, "SNAP.xlsx"), index=False)
print("Done.")

Total pairs: 15139 Index(['DRUGID', 'UNIPROTID'], dtype='object')
Validating SMILES...


[11:08:23] Unusual charge on atom 42 number of radical electrons set to zero


Valid pairs: 13871
Invalid SMILES pairs removed: 0
Done.


In [4]:
# 读取数据
pair_df = pd.read_excel(r'./final_data/SNAP.xlsx')
print('Before filtering, number of pairs:', len(pair_df))

# ===== 0. 统一 ID 类型，防止 merge / set 出错 =====
pair_df['DRUGID'] = pair_df['DRUGID'].str.strip()
pair_df['UNIPROTID'] = pair_df['UNIPROTID'].str.strip()

# ===== 1. 构建字典 =====
drug_df = pair_df[['DRUGID', 'INCHIKEY', 'DRUGNAME', 'SMILES']].drop_duplicates(subset=['DRUGID'])
protein_df = pair_df[["UNIPROTID", 'GENENAME', "SEQUENCE"]].drop_duplicates(subset=['UNIPROTID'])

drug_dict = dict(zip(drug_df['DRUGID'], drug_df['SMILES']))
protein_dict = dict(zip(protein_df['UNIPROTID'], protein_df['SEQUENCE']))

print('Number of drugs:', len(drug_dict))
print('Number of proteins:', len(protein_dict))

# 保存信息表
drug_df.to_excel(r'./final_data/DrugInfo.xlsx', index=False)
protein_df.to_excel(r'./final_data/ProtInfo.xlsx', index=False)

# ===== 2. 正样本 =====
pos_pairs = set(zip(pair_df['DRUGID'], pair_df['UNIPROTID']))
num_pos = len(pos_pairs)
print('Number of positive pairs:', num_pos)

# ===== 3. 生成全部候选负样本 =====
all_drugs = pair_df['DRUGID'].dropna().unique()
all_proteins = pair_df['UNIPROTID'].dropna().unique()

print('Generating all candidate negative pairs...')
all_pairs = set(product(all_drugs, all_proteins))
all_neg_pairs = list(all_pairs - pos_pairs)

print('Total candidate negative pairs:', len(all_neg_pairs))

# ===== 4. 按不同正负比例采样 =====
ratios = [1, 3, 5, 7, 10]
seeds = [1, 11, 111, 1111, 11111]

for ratio in ratios:
    save_dir = f'./final_data/1_{ratio}'
    os.makedirs(save_dir, exist_ok=True)

    num_neg = num_pos * ratio

    if num_neg > len(all_neg_pairs):
        raise ValueError(f'Not enough negative pairs for ratio {ratio}: need {num_neg}, but only {len(all_neg_pairs)} available.')

    for i, seed in enumerate(seeds):
        print(f'Generating ratio 1:{ratio}, Set{i+1}, seed={seed}...')

        random.seed(seed)
        sampled_neg = random.sample(all_neg_pairs, num_neg)

        pos_df = pd.DataFrame(list(pos_pairs), columns=['DRUGID', 'UNIPROTID'])
        pos_df['Label'] = 1

        neg_df = pd.DataFrame(sampled_neg, columns=['DRUGID', 'UNIPROTID'])
        neg_df['Label'] = 0

        dataset = pd.concat([pos_df, neg_df], ignore_index=True)
        dataset = dataset.sample(frac=1, random_state=seed).reset_index(drop=True)

        save_path = os.path.join(save_dir, f'Set{i+1}.xlsx')
        dataset.to_excel(save_path, index=False)

        print(f'Saved: {save_path} | Pos: {len(pos_df)} | Neg: {len(neg_df)} | Total: {len(dataset)}')

print('All datasets generated!')

Before filtering, number of pairs: 13871
Number of drugs: 4516
Number of proteins: 2115
Number of positive pairs: 13871
Generating all candidate negative pairs...
Total candidate negative pairs: 9537469
Generating ratio 1:1, Set1, seed=1...
Saved: ./final_data/1_1/Set1.xlsx | Pos: 13871 | Neg: 13871 | Total: 27742
Generating ratio 1:1, Set2, seed=11...
Saved: ./final_data/1_1/Set2.xlsx | Pos: 13871 | Neg: 13871 | Total: 27742
Generating ratio 1:1, Set3, seed=111...
Saved: ./final_data/1_1/Set3.xlsx | Pos: 13871 | Neg: 13871 | Total: 27742
Generating ratio 1:1, Set4, seed=1111...
Saved: ./final_data/1_1/Set4.xlsx | Pos: 13871 | Neg: 13871 | Total: 27742
Generating ratio 1:1, Set5, seed=11111...
Saved: ./final_data/1_1/Set5.xlsx | Pos: 13871 | Neg: 13871 | Total: 27742
Generating ratio 1:3, Set1, seed=1...
Saved: ./final_data/1_3/Set1.xlsx | Pos: 13871 | Neg: 41613 | Total: 55484
Generating ratio 1:3, Set2, seed=11...
Saved: ./final_data/1_3/Set2.xlsx | Pos: 13871 | Neg: 41613 | Total: 5